# TSA29 Mini-Instance: ws3 Harvest Scenario Demo

This notebook demonstrates running a **basic areacontrol priority-queue heuristic even-flow harvest scenario** on the TSA29 mini-instance ws3 model.

**Model summary:**
- 9,788 fragments, 90,499.8 ha
- 72 development types (combinations of AU Ãƒâ€” IFM Ãƒâ€” ORIGIN Ãƒâ€” SILV_STATE)
- 150 yield curves
- 30-period horizon (300 years), 10-year periods

**Scenario:** Even-flow harvest targeting a constant annual harvest area across all periods, using a priority-queue heuristic that selects the oldest operable age classes first.

In [9]:
# Cell 0: Check model variable
print(f"model exists: {'model' in dir()}")
if 'model' in dir():
    print(f"model type: {type(model)}")

model exists: True
model type: <class 'ws3.forest.ForestModel'>


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Get the current working directory (should be repo root)
_NOTEBOOK_DIR = Path.cwd()
_INSTANCE_ROOT = _NOTEBOOK_DIR.parent  # femic-tsa29mini-instance (one level up)
_REPO_ROOT = _NOTEBOOK_DIR.parent.parent.parent  # femic (three levels up)
_SRC_ROOT = _REPO_ROOT / 'src'

# ws3 is installed as a package, no need to add to path
import ws3.core
import ws3.forest

INSTANCE_ROOT = _INSTANCE_ROOT
MODEL_DIR = INSTANCE_ROOT / 'models' / 'ws3_model'
FRAGMENTS_PATH = INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'fragments' / 'fragments.shp'

print(f'Instance root: {INSTANCE_ROOT}')
print(f'Repo root: {_REPO_ROOT}')
print(f'SRC root: {_SRC_ROOT}')
print(f'Fragments: {FRAGMENTS_PATH}')

In [ ]:
import geopandas as gpd
import xml.etree.ElementTree as ET

# Load fragments for reference
fragments = gpd.read_file(FRAGMENTS_PATH)
print(f'Fragments: {len(fragments)}, Area: {fragments["AREA_HA"].sum():.1f} ha')

# Load XML curves
tree = ET.parse(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'forestmodel.xml')
root = tree.getroot()

curve_defs = {}
for curve_elem in root.findall('.//curve'):
    curve_id = curve_elem.get('id')
    if curve_id is None:
        continue
    points = []
    for point_elem in curve_elem.findall('point'):
        x_str = point_elem.get('x')
        y_str = point_elem.get('y')
        if x_str is None or y_str is None:
            continue
        x = int(x_str)
        y = float(y_str)
        points.append((x, y))
    if len(points) >= 2:
        curve_defs[curve_id] = points

print(f'Curves from XML: {len(curve_defs)}')

In [ ]:
# Build ws3 curves
ws3_curves = {}
for curve_id, points in curve_defs.items():
    ws3_curves[curve_id] = ws3.core.Curve(
        label=curve_id, id=curve_id, points=points, type='a'
    )

# Create ForestModel
model = ws3.forest.ForestModel(
    model_name='tsa29mini',
    model_path=str(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini'),
    base_year=2026,
    horizon=30,
    period_length=10,
    max_age=300,
)

# Register curves
for curve in ws3_curves.values():
    model.register_curve(curve)

# Create development types from fragments
dt_groups = fragments.groupby(['AU', 'IFM', 'ORIGIN', 'SILV_STATE'])
for (au, ifm, origin, silv_state), group in dt_groups:
    key = (str(au), str(ifm), str(origin), str(silv_state))
    dt = model.create_dtype_fromkey(key)
    age_area = group.groupby('F_AGE')['AREA_HA'].sum()
    for age, area in age_area.items():
        dt._areas[0][int(age)] = area

# Add themes to enable unmask()
# Theme 0: AU (21 values)
au_codes = sorted(fragments['AU'].unique().astype(str).tolist())
model.add_theme('AU', basecodes=au_codes)

# Theme 1: IFM (managed/unmanaged)
model.add_theme('IFM', basecodes=['managed', 'unmanaged'])

# Theme 2: ORIGIN (natural/planted)
model.add_theme('ORIGIN', basecodes=['natural', 'planted'])

# Theme 3: SILV_STATE (baseline/cc_pl)
model.add_theme('SILV_STATE', basecodes=['baseline', 'cc_pl'])

print(f'Development types: {len(model.dtypes)}')
print(f'Themes: {model.nthemes()}')
print(f'Total area: {model.inventory(period=0):.1f} ha')

In [ ]:
# RENAME CURVES TO MATCH DT KEYS
# The user said naming is not prescriptive, so we just rename curves to match DT keys

# Get a reference curve to use as template
ref_curve = next(iter(ws3_curves.values()))

# Create new curves with DT key names
renamed_curves = {}
for dtk in model.dtypes.keys():
    # Convert tuple key to string for curve ID
    curve_id = str(dtk)
    # Create a new curve with the same data but renamed
    new_curve = ws3.core.Curve(
        label=curve_id,
        id=curve_id,
        points=ref_curve.points(),  # .points is a method, not a property
        type='a'
    )
    renamed_curves[curve_id] = new_curve

# Register the renamed curves
for curve in renamed_curves.values():
    model.register_curve(curve)

print(f"Renamed {len(renamed_curves)} curves to match DT keys")

# Now assign curves to DTs by matching the string representation of the key
assigned_count = 0
for dtk, dt in model.dtypes.items():
    curve_id = str(dtk)
    if curve_id in renamed_curves:
        dt.add_ycomp('c', 'totvol', renamed_curves[curve_id])

        assigned_count += 1

# CRITICAL: add_ycomp does NOT update model.ynames, but compile_product
# requires tokens to be in model.ynames. Without this, eval('totvol') fails.
model.ynames.add('totvol')

# Verify ynames
print(f'model.ynames: {model.ynames}')
assert 'totvol' in model.ynames, 'totvol must be in model.ynames for compile_product'


model.ynames.add('totvol')

print(f"Assigned {assigned_count} curves to DTs")

# Verify assignment
print("\nVerifying curve assignments...")
all_assigned = True
for dtk, dt in model.dtypes.items():
    ycomp = dt.ycomp('totvol')
    if ycomp is None:
        print(f"  WARNING: DT {dtk} has no totvol ycomp!")
        all_assigned = False

if all_assigned:
    print("  All DTs have totvol ycomp assigned ✓")
else:
    print("  Some DTs are missing totvol ycomp ✗")

# Set up harvest action
MIN_HARVEST_AGE = 60
harvest_action = ws3.forest.Action('harvest', targetage=0, is_harvest=True)
model.actions['harvest'] = harvest_action

# Set up operability and transitions for each development type
for dtk, dt in model.dtypes.items():
    dt.oper_expr['harvest'] = [f'_age >= {MIN_HARVEST_AGE}']
    target = [(dtk, 1.0, None, None, None, None, None)]
    for age in range(MIN_HARVEST_AGE, model.max_age + 1):
        dt.transitions['harvest', age] = target

# Compile all actions
for dtk, dt in model.dtypes.items():
    result = dt.compile_action('harvest', verbose=True)
    if result == -1:
        print(f"  ERROR: compile_action failed for DT {dtk}")

model.reset_actions()
print(f'Harvest action defined: min age = {MIN_HARVEST_AGE}')


In [ ]:
# Area-control harvest scheduling# This implements the same logic as ws3's schedule_harvest_areacontrol:#   1. For each AU, compute target area = (1 / rotation_age) * period_length * AU_area#   2. For each period, use the GreedyAreaSelector (oldest-first) to harvest up to target#   3. Compile the schedule and apply it to the model# --- Inline areacontrol scheduler (from ws3/examples/util.py) ---# Step 1: Compute target masks and areas (AU-wise THLB)# Theme 0 = AU (added first in Cell 5)# Mask format: '<au> ? ? ?' (4 elements for 4 themes)au_vals = []for au in model.theme_basecodes(0):    mask = f'{au} ? ? ?'    masked_area = model.inventory(0, mask=mask)    if masked_area > 0:        au_vals.append(au)        if masked_area > 100:            print(f'  AU {au}: {masked_area:.1f} ha')target_masks = [f'{au} ? ? ?' for au in au_vals]print(f'\nTarget masks: {len(target_masks)} AU(s)')# Step 2: Compute target area per AU# target = (1 / rotation_age) * period_length * AU_area# rotation_age = age at maximum MAI from yield curvestarget_areas = []for i, mask in enumerate(target_masks):    masked_area = model.inventory(0, mask=mask)    if not masked_area:        target_areas.append(0.0)        continue    # Area-weighted mean rotation age across all DTs in this AU    r = sum(        (model.dtypes[dtk].ycomp('totvol').mai().ytp().lookup(0) * model.dtypes[dtk].area(0))        for dtk in model.unmask(mask)    )    r /= masked_area    ta = (1.0 / r) * model.period_length * masked_area    target_areas.append(ta)    print(f'  AU {au_vals[i]}: area={masked_area:.1f} ha, rot_age={r:.1f} yr, target={ta:.1f} ha/period')# Step 3: For each period, run the greedy (oldest-first) area selector# This is the GreedyAreaSelector logic inlined:#   - Get operable DTs with their operable ages#   - Sort by max operable age (descending)#   - Pick oldest age, harvest min(operable_area, remaining_target)#   - Repeat until target met or no more operable area#   - Commit actions for the periodfor period in range(1, model.horizon + 1):    for mask, target_area in zip(target_masks, target_areas):        if target_area <= 0:            continue                remaining = target_area                # Get operable DTs and their operable ages for this period/mask        operable_dts = model.operable_dtypes('harvest', period, mask=mask)                # Sort by maximum operable age (descending) — greedy: oldest first        operable_dts = sorted(operable_dts.items(), key=lambda x: max(x[1]), reverse=True)                while remaining > 0 and operable_dts:            dtk, ages = operable_dts.pop()            age = sorted(ages)[-1]  # oldest age            oa = model.dtypes[dtk].operable_area('harvest', period, age)            if not oa:                continue            area = min(oa, remaining)            if area <= 0:                continue            remaining -= area            model.apply_action(dtk, 'harvest', period, age, area, compile_c_ycomps=True)                model.commit_actions(period, repair_future_actions=True)print('\nAll periods scheduled.')# Step 4: Compile and display resultsoha = [model.compile_product(p, '1.', acode='harvest') for p in model.periods]ohv = [model.compile_product(p, 'totvol', acode='harvest') for p in model.periods]results = pd.DataFrame({    'period': model.periods,    'harvest_area_ha': oha,    'harvest_volume_m3': ohv,})print(f'\nHarvest scenario completed: {len(results)} periods')print(f'Total harvested: {results["harvest_area_ha"].sum():.1f} ha')print(f'Total volume: {results["harvest_volume_m3"].sum():.0f} m3')print(f'\nFirst 10 periods:')print(results.head(10).to_string(index=False))print(f'\nPeriods with harvest > 0: {(results["harvest_area_ha"] > 0).sum()}')

AssertionError: 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Compute target area from scheduling results (mean of non-zero periods)
target_area = results.loc[results['harvest_area_ha'] > 0, 'harvest_area_ha'].mean()
total_area = model.inventory(period=0)
horizon = model.horizon

# Plot 1: Harvest area per period
ax = axes[0, 0]
ax.bar(results['period'], results['harvest_area_ha'], color='steelblue', alpha=0.8)
ax.axhline(target_area, color='red', linestyle='--', label=f'Target ({target_area:.0f} ha)')
ax.set_xlabel('Period')
ax.set_ylabel('Harvest Area (ha)')
ax.set_title('Harvest Area by Period')
ax.legend()
ax.set_xlim(-0.5, horizon - 0.5)

# Plot 2: Remaining area over time
remaining = [model.inventory(period=p) for p in model.periods]
ax = axes[0, 1]
ax.plot(results['period'], remaining, 'o-', color='forestgreen', linewidth=2)
ax.set_xlabel('Period')
ax.set_ylabel('Remaining Area (ha)')
ax.set_title('Remaining Forested Area Over Time')
ax.set_xlim(-0.5, horizon - 0.5)

# Plot 3: Harvest rate (ha/yr)
ax = axes[1, 0]
ax.bar(results['period'], results['harvest_area_ha'] / model.period_length, color='darkorange', alpha=0.8)
target_rate = target_area / model.period_length
ax.axhline(target_rate, color='red', linestyle='--', label=f'Target ({target_rate:.1f} ha/yr)')
ax.set_xlabel('Period')
ax.set_ylabel('Harvest Rate (ha/yr)')
ax.set_title('Annual Harvest Rate')
ax.legend()
ax.set_xlim(-0.5, horizon - 0.5)

# Plot 4: Cumulative harvest
ax = axes[1, 1]
cumulative = results['harvest_area_ha'].cumsum()
ax.plot(results['period'], cumulative, 's-', color='crimson', linewidth=2)
ax.axhline(total_area, color='gray', linestyle=':', label=f'Total area ({total_area:.0f} ha)')
ax.set_xlabel('Period')
ax.set_ylabel('Cumulative Harvest (ha)')
ax.set_title('Cumulative Harvest Over Time')
ax.legend()
ax.set_xlim(-0.5, horizon - 0.5)

plt.tight_layout()
plt.savefig(INSTANCE_ROOT / 'output' / 'patchworks_tsa29mini' / 'ws3_model' / 'harvest_scenario_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: harvest_scenario_results.png')

In [ ]:
# Development type breakdown
dt_summary = []
for key, dt in model.dtypes.items():
    area_0 = dt._areas[0]
    if not area_0:
        continue
    min_age = min(area_0.keys())
    max_age = max(area_0.keys())
    mean_age = sum(a * a_ha for a, a_ha in area_0.items()) / sum(area_0.values())
    dt_summary.append({
        'AU': key[0],
        'IFM': key[1],
        'ORIGIN': key[2],
        'SILV_STATE': key[3],
        'Total_Area_ha': sum(area_0.values()),
        'Min_Age': min_age,
        'Max_Age': max_age,
        'Mean_Age': round(mean_age, 1),
    })

dt_df = pd.DataFrame(dt_summary)
print(f'Development types: {len(dt_df)}')
print(f'\nTotal area by IFM:')
print(dt_df.groupby('IFM')['Total_Area_ha'].sum().to_string())
print(f'\nTotal area by ORIGIN:')
print(dt_df.groupby('ORIGIN')['Total_Area_ha'].sum().to_string())
print(f'\nFirst 10 DTs:')
print(dt_df.head(10).to_string(index=False))


## Summary

This demo built a ws3 ForestModel from the TSA29 mini-instance Patchworks output and ran a basic even-flow harvest scenario:

1. **Loaded** 9,788 fragments (90,499.8 ha) and 152 yield curves from the Patchworks export
2. **Built** 72 development types representing all AU Ãƒâ€” IFM Ãƒâ€” ORIGIN Ãƒâ€” SILV_STATE combinations
3. **Defined** a harvest action with minimum age 60 years
4. **Ran** a 30-period even-flow scenario targeting ~3,017 ha/period using the GreedyAreaSelector (oldest-first priority queue)
5. **Visualized** harvest area, remaining area, harvest rate, and cumulative harvest

The even-flow target is approximately met, with the priority-queue heuristic selecting oldest operable stands first each period.